<center> <span style="color:indigo">Machine Learning e Inferencia Bayesiana</span> </center> 

<div style="text-align: center;">
<img src="https://upload.wikimedia.org/wikipedia/commons/2/2b/Centro_Universitario_del_Guadalajara_Logo.png" alt="Drawing" style="width: 800px;"/>
</div>

<center> <span style="color:DarkBlue">  Tema 3. Cálculo diferencial: Minimización de costos </span>  </center>
<center> <span style="color:Blue"> Profesor: M. en C. Iván A. Toledano Juárez </span>  </center>

# Optimización de funciones de una variable: Minimización de costos

## Introducción

Imagina que una empresa necesita comprar un insumo (producto P) durante los próximos 12 meses. Este insumo se puede adquirir con dos proveedores: **A** y **B**, y se planea mantener constante la proporción mensual comprada a cada uno.

La empresa dispone de datos históricos de precios por mes de ambos proveedores. El objetivo es encontrar la **combinación óptima de compras** (porcentaje a A y porcentaje a B) que **minimice la variación del costo total** durante el periodo.

## El problema

Supongamos que cada mes se compra la misma cantidad de producto $q$. Si los precios mensuales de los proveedores son $p_A^{(i)}$ y $ p_B^{(i)} $, y decidimos comprar una fracción $\omega$ del producto al proveedor A y $1 - \omega$ al proveedor B, entonces el **precio combinado** ese mes será:

\begin{equation}
f^{(i)}(\omega) = \omega \cdot p_A^{(i)} + (1 - \omega) \cdot p_B^{(i)}
\end{equation}

donde $i$ indica el mes (por ejemplo, enero, febrero, etc.).

Este tipo de problema es similar a los que se encuentran en **finanzas** o en una administración de algún **portafolio**, donde se busca decidir cómo distribuir una inversión entre diferentes opciones (en este caso, proveedores), usando precios históricos para tomar decisiones que maximicen el beneficio o minimicen el costo.

La idea principal es la siguiente:

1. Para cada mes, calculamos el **precio promedio combinado** entre los proveedores, usando un valor dado de $\omega$ (el porcentaje que se le compra al proveedor A).
2. Con lo anterior, obtenemos una lista de precios combinados, uno por cada mes.
3. Calculamos el **promedio total** de esos precios.
4. Queremos encontrar el valor de $\omega$ que haga que los precios mensuales combinados **se desvíen lo menos posible de este promedio**. Es decir, que el costo sea estable y predecible.

Como las diferencias pueden ser tanto positivas como negativas, y se pueden cancelar entre sí, una estrategia común es elevar al cuadrado las diferencias (varianza, para que todas sean positivas), y luego tomar el promedio de estas desviaciones al cuadrado.

Así, la función que nos interesa construir y minimizar es la **varianza** del costo total (o su aproximación) como función de $\omega$:

\begin{equation}
\mathcal{L}(\omega) = \frac{1}{n} \sum_{i=1}^{n} \left( f^{(i)}(\omega) - \bar{f}(\omega) \right)^2
\end{equation}

donde $\bar{f}(\omega)$ es el promedio de los costos $f^{(i)}(\omega)$.

Minimizar $\mathcal{L}(\omega)$ nos permite encontrar el valor de $\omega$ que **genera los costos más estables** durante el periodo analizado, lo cual es útil para presupuestar y negociar con los proveedores.

## Objetivo de la práctica

- Construir la función $\mathcal{L}(\omega)$ a partir de datos históricos.
- Visualizar su comportamiento para distintos valores de $\omega$.
- Encontrar el valor de $\omega$ que minimiza $\mathcal{L}(\omega)$.
- Calcular la derivada de la función de forma numérica usando `np.gradient` y verificar que en el mínimo se cumple $\mathcal{L}'(\omega) \approx 0$.

Esta práctica nos permitirá **aplicar conceptos de cálculo diferencial y optimización** a un problema realista con datos reales, usando Python y herramientas de análisis numérico.

---


In [1]:
#agregamos las librerias
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sympy import *
from sympy.utilities.lambdify import lambdify

In [4]:
#Importamos el los datos
df_prices = pd.read_csv("data/prices.csv")
df_prices['date'] = pd.to_datetime(df_prices['date'], format="%d/%m/%Y")
df_prices.head(10)

,date,price_supplier_a_dollars_per_item,price_supplier_b_dollars_per_item
0,2016-02-01,104,76
1,2016-03-01,108,76
2,2016-04-01,101,84
3,2016-05-01,104,79
4,2016-06-01,102,81
5,2016-07-01,105,84
6,2016-08-01,114,90
7,2016-09-01,102,93
8,2016-10-01,105,93
9,2016-11-01,101,99


In [5]:
df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   date                               50 non-null     datetime64[ns]
 1   price_supplier_a_dollars_per_item  50 non-null     int64         
 2   price_supplier_b_dollars_per_item  50 non-null     int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 1.3 KB


In [7]:
p_a = df_prices['price_supplier_a_dollars_per_item'].to_numpy()
p_b = df_prices['price_supplier_b_dollars_per_item'].to_numpy()